# LAB 02 — Core Implementation (Bài 14 & 15)

- Bài 14: tự cài đặt `build_vocabulary`, `count_ngrams`, `train_unigram/bigram/trigram`, `probability`, `sentence_probability`, `next_word_distribution` (class `NGramLanguageModel`).
- Bài 15: `sentence_log_probability`.

In [1]:
import math
import re
import gzip
import json
from collections import Counter, defaultdict

SOS = "<s>"
EOS = "</s>"


def split_sentences(text):
    sentences = re.split(r"[.!?]+", text)
    return [s.strip() for s in sentences if s.strip()]


def tokenize(sentence):
    return re.findall(r"[a-z\u00e0-\u1ef9\u0300-\u036f0-9]+", sentence.lower())


def load_documents(path, text_field="text", limit=None):
    opener = gzip.open if path.endswith(".gz") else open
    docs = []
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            docs.append(json.loads(line)[text_field] if ".json" in path else line)
            if limit is not None and len(docs) >= limit:
                break
    return docs


def prepare_sentences(docs):
    """docs -> list cac cau, moi cau la list token."""
    sentences = []
    for doc in docs:
        for s in split_sentences(doc):
            tokens = tokenize(s)
            if tokens:
                sentences.append(tokens)
    return sentences

## 14. Core implementation

In [2]:
def build_vocabulary(sentences, min_count=1):
    """Tap tu vung (gom ca </s>). <s> chi la context nen khong dua vao vocab."""
    counts = Counter(w for s in sentences for w in s)
    vocab = {w for w, c in counts.items() if c >= min_count}
    vocab.add(EOS)
    return vocab


def pad_sentence(tokens, n):
    return [SOS] * (n - 1) + list(tokens) + [EOS]


def count_ngrams(sentences, n):
    """Dem cac n-gram (co padding <s>, </s>)."""
    counts = Counter()
    for tokens in sentences:
        padded = pad_sentence(tokens, n)
        for i in range(len(padded) - n + 1):
            counts[tuple(padded[i:i + n])] += 1
    return counts


def count_contexts(ngram_counts):
    """C(h): tong count cac n-gram co cung context h (n-1 tu dau)."""
    contexts = Counter()
    for ngram, c in ngram_counts.items():
        contexts[ngram[:-1]] += c
    return contexts


def train_ngram(sentences, n):
    ngram_counts = count_ngrams(sentences, n)
    return ngram_counts, count_contexts(ngram_counts)


def train_unigram(sentences):
    return train_ngram(sentences, 1)   # context = () , C(()) = N


def train_bigram(sentences):
    return train_ngram(sentences, 2)


def train_trigram(sentences):
    return train_ngram(sentences, 3)

In [3]:
class NGramLanguageModel:
    def __init__(self, n, smoothing="none"):
        assert n in (1, 2, 3), "n phai la 1, 2 hoac 3"
        assert smoothing in ("none", "laplace")
        self.n = n
        self.smoothing = smoothing

    # ---------- training ----------
    def fit(self, sentences):
        self.vocab = build_vocabulary(sentences)
        trainer = {1: train_unigram, 2: train_bigram, 3: train_trigram}[self.n]
        self.ngram_counts, self.context_counts = trainer(sentences)
        # chi muc: context -> {tu tiep theo: count}, dung cho next_word_distribution
        self.followers = defaultdict(dict)
        for ngram, c in self.ngram_counts.items():
            self.followers[ngram[:-1]][ngram[-1]] = c
        return self

    # ---------- helpers ----------
    @staticmethod
    def _to_tokens(x):
        return tokenize(x) if isinstance(x, str) else list(x)

    def _normalize_context(self, context):
        """Lay n-1 tu cuoi cua context, them <s> neu context ngan hon."""
        if self.n == 1:
            return ()
        ctx = self._to_tokens(context)[-(self.n - 1):]
        ctx = [SOS] * (self.n - 1 - len(ctx)) + ctx
        return tuple(ctx)

    # ---------- probability ----------
    def probability(self, context, word):
        """P(word | context)."""
        ctx = self._normalize_context(context)
        count = self.ngram_counts.get(ctx + (word,), 0)
        ctx_count = self.context_counts.get(ctx, 0)
        if self.smoothing == "laplace":
            return (count + 1) / (ctx_count + len(self.vocab))
        if ctx_count == 0:
            return 0.0
        return count / ctx_count

    def sentence_probability(self, sentence):
        """P(S) = tich cac P(w_t | context). Co the bang 0 (MLE) hoac underflow voi cau dai."""
        padded = pad_sentence(self._to_tokens(sentence), self.n)
        prob = 1.0
        for i in range(self.n - 1, len(padded)):
            prob *= self.probability(padded[i - (self.n - 1):i], padded[i])
        return prob

    def sentence_log_probability(self, sentence):
        """log P(S) = tong log P(w_t | context). Tra ve -inf neu co xac suat bang 0."""
        padded = pad_sentence(self._to_tokens(sentence), self.n)
        total = 0.0
        for i in range(self.n - 1, len(padded)):
            p = self.probability(padded[i - (self.n - 1):i], padded[i])
            if p == 0.0:
                return float("-inf")
            total += math.log(p)
        return total

    def next_word_distribution(self, context, top_k=None):
        """Tra ve list (word, prob) giam dan theo prob."""
        ctx = self._normalize_context(context)
        if self.smoothing == "laplace":
            dist = [(w, self.probability(ctx, w)) for w in self.vocab]
        else:
            ctx_count = self.context_counts.get(ctx, 0)
            if ctx_count == 0:
                return []
            dist = [(w, c / ctx_count) for w, c in self.followers[ctx].items()]
        dist.sort(key=lambda x: x[1], reverse=True)
        return dist[:top_k] if top_k else dist

### Kiểm thử trên corpus nhỏ (corpus ở Bài 1)

In [4]:
toy_text = ["the cat eats fish", "the cat likes fish", "the dog eats meat"]
toy = [tokenize(s) for s in toy_text]

uni = NGramLanguageModel(1).fit(toy)
bi = NGramLanguageModel(2).fit(toy)
tri = NGramLanguageModel(3).fit(toy)

print("Vocabulary:", sorted(bi.vocab))
print("Bigram counts:", dict(bi.ngram_counts))
print()

# Bigram: P(cat|the)=2/3, P(dog|the)=1/3, P(eats|cat)=1/2, P(likes|cat)=1/2
for ctx, w in [("the", "cat"), ("the", "dog"), ("cat", "eats"), ("cat", "likes")]:
    print(f"P({w}|{ctx}) = {bi.probability([ctx], w):.4f}")

# Kiem tra tong xac suat cac tu dung sau context = 1
for ctx in [["the"], ["cat"], ["eats"]]:
    total = sum(bi.probability(ctx, w) for w in bi.vocab)
    print(f"sum_w P(w|{ctx[0]}) = {total:.4f}")
    assert abs(total - 1) < 1e-9

print()
for s in ["the cat eats fish", "the dog eats fish", "the fish eats cat"]:
    print(f"{s!r}")
    for name, lm in [("unigram", uni), ("bigram", bi), ("trigram", tri)]:
        print(f"   {name:8s} P = {lm.sentence_probability(s):.6f} | log P = {lm.sentence_log_probability(s)}")

print()
print("next word sau 'the cat' (bigram):", bi.next_word_distribution("the cat"))
print("next word sau 'the cat' (trigram):", tri.next_word_distribution("the cat"))

Vocabulary: ['</s>', 'cat', 'dog', 'eats', 'fish', 'likes', 'meat', 'the']
Bigram counts: {('<s>', 'the'): 3, ('the', 'cat'): 2, ('cat', 'eats'): 1, ('eats', 'fish'): 1, ('fish', '</s>'): 2, ('cat', 'likes'): 1, ('likes', 'fish'): 1, ('the', 'dog'): 1, ('dog', 'eats'): 1, ('eats', 'meat'): 1, ('meat', '</s>'): 1}

P(cat|the) = 0.6667
P(dog|the) = 0.3333
P(eats|cat) = 0.5000
P(likes|cat) = 0.5000
sum_w P(w|the) = 1.0000
sum_w P(w|cat) = 1.0000
sum_w P(w|eats) = 1.0000

'the cat eats fish'
   unigram  P = 0.000095 | log P = -9.263584886494995
   bigram   P = 0.166667 | log P = -1.791759469228055
   trigram  P = 0.333333 | log P = -1.0986122886681098
'the dog eats fish'
   unigram  P = 0.000047 | log P = -9.95673206705494
   bigram   P = 0.166667 | log P = -1.791759469228055
   trigram  P = 0.000000 | log P = -inf
'the fish eats cat'
   unigram  P = 0.000095 | log P = -9.263584886494995
   bigram   P = 0.000000 | log P = -inf
   trigram  P = 0.000000 | log P = -inf

next word sau 'the cat

## 15. Log probability



In [5]:

long_sentence = ["the"] * 2000

p = uni.sentence_probability(long_sentence)
logp = uni.sentence_log_probability(long_sentence)

print("Tich xac suat truc tiep :", p)       # 0.0 do underflow
print("Log probability          :", logp)   # gia tri huu han
print("Gia tri thuc te 0.2^2000 ~ 10^", 2000 * math.log10(0.2))

Tich xac suat truc tiep : 0.0
Log probability          : -3220.485262780719
Gia tri thuc te 0.2^2000 ~ 10^ -1397.9400086720375


## Chạy thử trên corpus C4 (Google Drive)

In [6]:
from google.colab import drive
drive.mount('/content/drive')

CORPUS_PATH = "/content/drive/MyDrive/NLP/lab02/c4-train.00000-of-01024-30K.json.gz"
LIMIT = None

Mounted at /content/drive


In [7]:
docs = load_documents(CORPUS_PATH, limit=LIMIT)
sentences = prepare_sentences(docs)
print(f"{len(docs)} documents, {len(sentences)} sentences")

models = {n: NGramLanguageModel(n).fit(sentences) for n in (1, 2, 3)}
for n, lm in models.items():
    print(f"n={n}: {len(lm.ngram_counts):,} unique n-grams, vocab = {len(lm.vocab):,}")

30000 documents, 685841 sentences
n=1: 190,836 unique n-grams, vocab = 190,836
n=2: 2,847,910 unique n-grams, vocab = 190,836
n=3: 7,042,586 unique n-grams, vocab = 190,836


In [8]:
for context in ["the", "natural language", "machine learning"]:
    print(f"\nContext: {context!r}")
    for name, n in [("bigram", 2), ("trigram", 3)]:
        top = models[n].next_word_distribution(context, top_k=5)
        print(f"  {name}: " + ", ".join(f"{w} {p:.3f}" for w, p in top))

s = "this is a good product"
for n in (1, 2, 3):
    print(f"n={n}: log P({s!r}) = {models[n].sentence_log_probability(s):.3f}")


Context: 'the'
  bigram: first 0.010, most 0.010, best 0.010, same 0.009, world 0.008
  trigram: first 0.015, new 0.009, company 0.008, most 0.008, best 0.007

Context: 'natural language'
  bigram: </s> 0.127, and 0.103, of 0.055, is 0.032, to 0.022
  trigram: processing 0.375, </s> 0.188, laboratory 0.188, toolkit 0.062, apps 0.062

Context: 'machine learning'
  bigram: </s> 0.090, and 0.080, to 0.061, about 0.042, how 0.032
  trigram: and 0.141, </s> 0.087, to 0.054, based 0.054, models 0.054
n=1: log P('this is a good product') = -31.582
n=2: log P('this is a good product') = -21.480
n=3: log P('this is a good product') = -inf
